# Run analysis: test-period predictions across rolling windows

Reads the run's `results_lgbm.json`, `predictions/`, and `curves/`; saves every
figure to `plots/analysis_*.png`. Re-run top to bottom for any run folder.

Sections:
1. Label distribution (scale check for issue I-8)
2. Daily metrics: Pearson IC, Rank IC (Spearman), daily Kendall
3. Per-window summary table: ICIR, Newey–West t-stat, train/val/test R²
4. Plots: IC over time, cumulative IC, per-window metrics, R² by window,
   quintile returns, calibration


In [ ]:
import json
import re
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from scipy import stats

# The notebook lives in the run folder; VS Code runs it from there.
RUN_DIR = Path.cwd()
assert (RUN_DIR / "results_lgbm.json").exists(), f"Not a run folder: {RUN_DIR}"
DATA_FILE = RUN_DIR.parents[1] / "data.parquet"
PLOTS_DIR = RUN_DIR / "plots"
PLOTS_DIR.mkdir(exist_ok=True)

# One fixed colour per window (categorical, fixed order, never cycled).
WINDOW_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]
SPLIT_COLORS = {"train": "#52514e", "val": "#6250d6", "test": "#e34948"}
plt.rcParams.update({
    "figure.dpi": 100,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "lines.linewidth": 2,
    "legend.frameon": False,
})
warnings.filterwarnings("ignore", message=".*non-interactive.*")


def save(fig, name: str) -> None:
    """Save a figure to plots/analysis_<name>.png, then show it."""
    fig.savefig(PLOTS_DIR / f"analysis_{name}.png", dpi=120, bbox_inches="tight")
    plt.show()


In [ ]:
results = json.loads((RUN_DIR / "results_lgbm.json").read_text())
test_start_to_window = {r["test_start"]: k + 1 for k, r in enumerate(results)}


def window_of(path: Path) -> int:
    """Window number (1-based) from the test start date in a file name."""
    return test_start_to_window[re.search(r"_test_(\d{4}-\d{2}-\d{2})_", path.name)[1]]


preds = pl.concat([
    pl.read_parquet(f).with_columns(pl.lit(window_of(f)).alias("window"))
    for f in sorted((RUN_DIR / "predictions").glob("predictions_*.parquet"))
]).sort(["window", "Date", "Code"])
curves = {
    window_of(f): json.loads(f.read_text())
    for f in sorted((RUN_DIR / "curves").glob("curves_*.json"))
}
n_windows = len(results)
assert preds["window"].n_unique() == n_windows == len(curves)
print(f"{n_windows} windows, {preds.height:,} test rows, "
      f"{preds['Date'].n_unique()} test dates "
      f"({preds['Date'].min()} -> {preds['Date'].max()})")


## 1. Label distribution

The label's spread and tails, in label units (issue I-8). The robust std
(1.4826 × MAD) ignores the tails.


In [ ]:
labels = pl.read_parquet(DATA_FILE, columns=["Date", "label"]).drop_nulls()
y = labels["label"].to_numpy()
mad_std = 1.4826 * np.median(np.abs(y - np.median(y)))
qs = [0.001, 0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99, 0.999]
print(f"rows={len(y):,}  mean={y.mean():.5f}  std={y.std():.5f}  "
      f"robust std={mad_std:.5f}  min={y.min():.3f}  max={y.max():.3f}")
print("quantiles:", {q: round(float(v), 4) for q, v in zip(qs, np.quantile(y, qs))})
print(f"excess kurtosis={stats.kurtosis(y):.1f} (0 for a normal distribution)")

per_day = labels.group_by("Date").agg(
    pl.col("label").mean().alias("mean"),
    pl.col("label").std().alias("std"),
).sort("Date")
print(f"daily cross-sectional mean: avg={per_day['mean'].mean():.5f}, "
      f"std over days={per_day['mean'].std():.5f}")
print(f"daily cross-sectional std: median={per_day['std'].median():.5f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 3.6))
axes[0].hist(y, bins=200, color=WINDOW_COLORS[0])
axes[0].set_yscale("log")
axes[0].set_title("label histogram (log count)")
axes[0].set_xlabel("label")
dates = per_day["Date"].to_numpy()
axes[1].plot(dates, per_day["std"], color=WINDOW_COLORS[0], lw=1, label="cross-sectional std")
axes[1].plot(dates, per_day["mean"], color=WINDOW_COLORS[1], lw=1, label="cross-sectional mean")
axes[1].axhline(0, color="black", lw=0.8)
axes[1].set_title("label per date")
axes[1].tick_params(axis="x", labelrotation=30)
axes[1].legend()
save(fig, "label_distribution")


## 2. Daily metrics

Every metric is computed **within each test date** (stocks vs stocks on the
same day), then summarised per window.


In [ ]:
daily = (
    preds.group_by(["window", "Date"], maintain_order=True)
    .agg(
        pl.corr("Prediction", "label").alias("ic"),
        pl.corr("Prediction", "label", method="spearman").alias("rank_ic"),
        pl.len().alias("n"),
    )
    .sort(["window", "Date"])
)
kendall = [
    stats.kendalltau(d["Prediction"].to_numpy(), d["label"].to_numpy()).statistic
    for d in preds.partition_by(["window", "Date"], maintain_order=True)
]
daily = daily.with_columns(pl.Series("kendall", kendall))

# Quintiles: rank stocks by prediction within each date; 1 = lowest, 5 = highest.
key = ["window", "Date"]
preds = preds.with_columns(
    ((pl.col("Prediction").rank("ordinal").over(key).cast(pl.Int64) - 1) * 5
     // pl.len().over(key) + 1).alias("quintile")
)
quint_daily = preds.group_by(["window", "Date", "quintile"]).agg(
    pl.col("label").mean().alias("label_mean")
)
spread = (
    quint_daily.pivot(on="quintile", index=["window", "Date"], values="label_mean")
    .select("window", "Date", (pl.col("5") - pl.col("1")).alias("q5_minus_q1"))
)
daily = daily.join(spread, on=["window", "Date"]).sort(["window", "Date"])
daily.head()


## 3. Per-window summary

- **ICIR** = mean(IC) / std(IC), unscaled, so it is comparable across windows
  (the script's `ir` multiplies by √days).
- **t (naive)** assumes independent days. **t (NW)** uses Newey–West standard
  errors with 4 lags, because 5-day labels overlap, making neighbouring days'
  ICs correlated.
- **IC lag-1 autocorr**: how correlated consecutive days' ICs are.
- **val R² / train R²** at `best_iteration`, from the learning curves; test R²
  from the results. val ≫ test would indicate the I-1 leak.
- **Q5−Q1**: mean label of the top-quintile stocks minus the bottom quintile,
  averaged over days. A gross spread in label units, not a profit (overlapping
  horizons, no costs).


In [ ]:
def newey_west_t(x: np.ndarray, lags: int = 4) -> float:
    """t-statistic of mean(x) with Newey–West (Bartlett-weighted) standard error."""
    e = x - x.mean()
    n = len(x)
    var = e @ e / n
    for lag in range(1, lags + 1):
        var += 2 * (1 - lag / (lags + 1)) * (e[lag:] @ e[:-lag]) / n
    return float(x.mean() / np.sqrt(var / n))


rows = []
for w in range(1, n_windows + 1):
    d = daily.filter(pl.col("window") == w)
    ic = d["ic"].drop_nans().to_numpy()
    r = results[w - 1]
    best = curves[w]["best_iteration"]
    rows.append({
        "window": w,
        "test": f"{r['test_start']} -> {r['test_end']}",
        "days": len(ic),
        "features": r["features"],
        "best_iter": r["best_iteration"],
        "IC": ic.mean(),
        "RankIC": d["rank_ic"].mean(),
        "Kendall(daily)": d["kendall"].mean(),
        "kt(pooled)": r["kt"],
        "ICIR": ic.mean() / ic.std(ddof=1),
        "IC>0 %": 100 * (ic > 0).mean(),
        "t(naive)": ic.mean() / ic.std(ddof=1) * np.sqrt(len(ic)),
        "t(NW)": newey_west_t(ic),
        "script ir": r["ir"],
        "IC ac1": np.corrcoef(ic[1:], ic[:-1])[0, 1],
        "train R2": curves[w]["curves"]["train"]["r2"][best - 1],
        "val R2": curves[w]["curves"]["val"]["r2"][best - 1],
        "test R2": r["r2"],
        "Q5-Q1": d["q5_minus_q1"].mean(),
        "runtime s": r["runtime"],
        "peak GB": r["peak_memory_GB"],
    })
summary = pl.DataFrame(rows)
all_ic = daily["ic"].drop_nans().to_numpy()
print(f"All windows: IC={all_ic.mean():.4f}  RankIC={daily['rank_ic'].mean():.4f}  "
      f"ICIR={all_ic.mean() / all_ic.std(ddof=1):.2f}  t(NW)={newey_west_t(all_ic):.1f}  "
      f"IC>0 on {100 * (all_ic > 0).mean():.0f}% of days")
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=4, tbl_width_chars=250):
    display(summary)  # noqa: F821  (IPython built-in)


## 4. Plots


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6.5), sharex=True, sharey=True)
for ax, col, name in zip(axes, ["ic", "rank_ic"], ["Pearson IC", "Rank IC (Spearman)"]):
    for w in range(1, n_windows + 1):
        d = daily.filter(pl.col("window") == w)
        x = d["Date"].to_numpy()
        ax.plot(x, d[col], color="#b5b4ad", lw=0.7)
        ax.plot(x, d[col].rolling_mean(20, min_samples=5),
                color=WINDOW_COLORS[w - 1], label=f"W{w} (20-day mean)")
    ax.axhline(0, color="black", lw=0.8)
    ax.set_title(f"Daily {name}: grey = daily, colour = 20-day rolling mean per window")
axes[0].legend(ncol=n_windows, loc="upper left", fontsize=8)
save(fig, "ic_over_time")

fig, ax = plt.subplots(figsize=(14, 3.8))
x = daily["Date"].to_numpy()
ax.plot(x, daily["ic"].fill_nan(0).cum_sum(), color=WINDOW_COLORS[0], label="Pearson IC")
ax.plot(x, daily["rank_ic"].fill_nan(0).cum_sum(), color=WINDOW_COLORS[1], label="Rank IC")
for w in range(2, n_windows + 1):
    ax.axvline(daily.filter(pl.col("window") == w)["Date"][0], color="gray", ls="--", lw=0.8)
ax.set_title("Cumulative daily IC (slope = IC level; not a profit curve). Dashed: new window/model")
ax.legend()
save(fig, "cumulative_ic")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6), sharey=True)
wins = summary["window"].to_list()
for ax, col in zip(axes, ["IC", "RankIC", "Kendall(daily)"]):
    ax.bar(wins, summary[col], color=WINDOW_COLORS[:n_windows], width=0.7)
    ax.set_title(f"mean daily {col.replace('(daily)', '')}")
    ax.set_xticks(wins, [f"W{w}" for w in wins])
save(fig, "metrics_by_window")

fig, ax = plt.subplots(figsize=(8, 3.8))
for k, (split, col) in enumerate([("train", "train R2"), ("val", "val R2"), ("test", "test R2")]):
    ax.plot(wins, summary[col], "o-", color=SPLIT_COLORS[split], label=split, ms=8)
ax.set_xticks(wins, [f"W{w}" for w in wins])
ax.set_title("R² by window (train/val at best_iteration)")
ax.legend()
save(fig, "r2_by_window")


In [ ]:
quint = (
    quint_daily.group_by(["window", "quintile"]).agg(pl.col("label_mean").mean())
    .sort(["window", "quintile"])
)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for w in range(1, n_windows + 1):
    q = quint.filter(pl.col("window") == w)
    axes[0].plot(q["quintile"], q["label_mean"], "o-", color=WINDOW_COLORS[w - 1], label=f"W{w}", ms=6)
axes[0].axhline(0, color="black", lw=0.8)
axes[0].set_xticks([1, 2, 3, 4, 5])
axes[0].set_xlabel("prediction quintile (1 = lowest predicted)")
axes[0].set_title("Mean label by prediction quintile")
axes[0].legend(ncol=2)

# Calibration: 20 prediction bins per window; perfect calibration lies on y = x.
binned = (
    preds.with_columns(
        ((pl.col("Prediction").rank("ordinal").over("window").cast(pl.Int64) - 1) * 20
         // pl.len().over("window")).alias("bin")
    )
    .group_by(["window", "bin"]).agg(pl.col("Prediction").mean(), pl.col("label").mean())
    .sort(["window", "bin"])
)
for w in range(1, n_windows + 1):
    b = binned.filter(pl.col("window") == w)
    axes[1].plot(b["Prediction"], b["label"], "o-", color=WINDOW_COLORS[w - 1], label=f"W{w}", ms=4, lw=1.5)
lim = [binned["Prediction"].min(), binned["Prediction"].max()]
axes[1].plot(lim, lim, "--", color="gray", lw=1, label="y = x")
axes[1].set_xlabel("mean prediction in bin")
axes[1].set_ylabel("mean label in bin")
axes[1].set_title("Calibration (20 prediction bins per window)")
axes[1].legend(ncol=2, fontsize=8)
save(fig, "quintiles_calibration")
print(quint.pivot(on="quintile", index="window", values="label_mean"))


## 5. Finer grain: months and robust summaries

A weak window can mean broad weakness, or a few bad stretches inside an
otherwise good period. Section 5 separates the two.

**How to read it.** A single month (~20 test days, with strongly
autocorrelated daily ICs) is worth only ~5 independent days. One bad month
proves little; look for **patterns**: months that are bad across several
models, or runs of weak months. Headline metrics stay full-period. Months are
never excluded after the fact (that would be cherry-picking).

- **Top panel:** monthly mean daily **Rank IC** (bars, colour = window) and
  **Pearson IC** (black dots). Dots far above their bar mark months where the
  signal worked only through a few extreme stocks.
- **Bottom panel:** monthly **calibration slope**, the regression slope of
  label on prediction. 1 means magnitudes are right; below 1 means
  overconfident (predictions too spread out); below 0.5 (dotted) means
  strongly overconfident.
- **Robust table:** median daily IC and a 10%-trimmed mean. If they are close
  to the mean, the window's result isn't driven by a few extreme days.


In [ ]:
def month_stats(frame: pl.DataFrame, by: list[str]) -> pl.DataFrame:
    """Pooled R² and calibration slope per group of rows."""
    return frame.group_by(by).agg(
        (1 - ((pl.col("label") - pl.col("Prediction")) ** 2).sum()
         / ((pl.col("label") - pl.col("label").mean()) ** 2).sum()).alias("R2"),
        (pl.cov("Prediction", "label") / pl.col("Prediction").var()).alias("slope"),
    )


preds = preds.with_columns(pl.col("Date").dt.strftime("%Y-%m").alias("month"))
daily = daily.with_columns(pl.col("Date").dt.strftime("%Y-%m").alias("month"))
monthly = (
    daily.group_by(["window", "month"]).agg(
        pl.len().alias("days"),
        pl.col("ic").mean().alias("IC"),
        pl.col("rank_ic").mean().alias("RankIC"),
        (100 * (pl.col("ic") > 0).mean()).alias("IC>0 %"),
    )
    .join(month_stats(preds, ["window", "month"]), on=["window", "month"])
    .sort(["month", "window"])
)
with pl.Config(tbl_rows=-1, float_precision=3):
    display(monthly)  # noqa: F821

x = np.arange(monthly.height)
colors = [WINDOW_COLORS[w - 1] for w in monthly["window"]]
labels_x = [m if d >= 5 else f"{m}*" for m, d in zip(monthly["month"], monthly["days"])]
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].bar(x, monthly["RankIC"], color=colors, width=0.8)
axes[0].plot(x, monthly["IC"], "o", color="#0b0b0b", ms=4, label="Pearson IC")
axes[0].axhline(0, color="black", lw=0.8)
axes[0].axhline(daily["rank_ic"].mean(), color="gray", ls="--", lw=1,
                label=f"overall mean Rank IC ({daily['rank_ic'].mean():.3f})")
axes[0].set_ylabel("correlation")
axes[0].set_title("Monthly mean daily Rank IC (bars, colour = window) and Pearson IC (dots)", pad=20)
axes[0].legend(loc="upper left", fontsize=8)
axes[1].bar(x, monthly["slope"], color=colors, width=0.8)
axes[1].axhline(1, color="gray", ls="--", lw=1, label="calibrated (slope 1)")
axes[1].axhline(0.5, color="gray", ls=":", lw=1, label="strongly overconfident (< 0.5)")
axes[1].set_ylabel("slope of label on prediction")
axes[1].set_title("Monthly calibration slope (< 1 = predictions too spread out)")
axes[1].legend(loc="upper right", fontsize=8)
axes[1].set_xticks(x, labels_x, rotation=60, fontsize=8)
# Window labels above the top panel.
for w in range(1, n_windows + 1):
    idx = np.flatnonzero(monthly["window"].to_numpy() == w)
    axes[0].annotate(f"W{w}", (idx.mean(), 1.0), xycoords=("data", "axes fraction"),
                     ha="center", va="bottom", fontsize=9, color=WINDOW_COLORS[w - 1])
fig.text(0.01, 0.0, "* fewer than 5 test days in that month for that window", fontsize=8)
save(fig, "monthly")

robust = daily.group_by("window").agg(
    pl.col("ic").mean().alias("IC mean"),
    pl.col("ic").median().alias("IC median"),
    pl.col("ic").filter(
        pl.col("ic").is_between(pl.col("ic").quantile(0.1), pl.col("ic").quantile(0.9))
    ).mean().alias("IC trimmed 10%"),
    pl.col("rank_ic").median().alias("RankIC median"),
    (100 * (pl.col("rank_ic") > 0).mean()).alias("RankIC>0 %"),
).sort("window")
with pl.Config(float_precision=4):
    display(robust)  # noqa: F821


## 6. Ex-ante regimes: does performance depend on observable conditions?

**What a regime split is.** Each test day is put into a group ("regime")
according to the market conditions on that day, and performance is compared
across groups.

**Why ex-ante (known in advance).** A regime split is only useful if you could
have *acted* on it: trust or size the signal more in some conditions, add the
condition as a model feature, or check whether a model variant (e.g. history
features) helps mainly in certain conditions. So each day's regime is built
**only from information available at the close of day T**. Labelling months
after the fact ("that was a crash month") can explain the past, but nobody
could have traded on it, and it invites cherry-picking. A regime split does
**not** try to foresee shocks. Finding that the regime *can't* flag the bad
periods is itself a useful (negative) result.

**How the regime is built.**
1. *Dispersion* = the cross-sectional std of the label on a date: how
   differently stocks moved. Labels are factor-adjusted, so this is
   stock-specific dispersion, **not** market volatility.
2. *Trailing dispersion* at T = the mean dispersion over dates T−26 … T−7.
   Those labels are fully realized by day T, so there is no look-ahead.
3. *Percentile* = where today's trailing dispersion ranks among the previous
   ~250 trading days (one year, at least 120 needed). Below the 33rd
   percentile is **calm**, above the 67th is **turbulent**, and in between is
   **normal**. Relative to the past year, so the thresholds adapt over time.

**How to read the charts.**
- *Timeline* (three rows sharing the date axis):
  1. Daily dispersion (thin) and the trailing measure (bold). Dotted lines
     mark the largest dispersion spikes. Note how the trailing measure reacts
     only after a spike: shock onsets look calm ex-ante.
  2. The percentile and the 1/3 and 2/3 thresholds. Background shading =
     regime (light = calm, dark = turbulent), the same in all rows.
  3. Daily Rank IC (faint) and its 20-day mean (bold), against the regime
     shading.
- *Summary*: left, mean daily Rank IC per regime, with Newey–West 95%
  confidence intervals and the number of days and of distinct **episodes**
  (separate runs of ≥ 5 days). Few episodes means weak evidence, however many
  days they cover. Right, the same per window: if the effect is real it
  should appear in most windows, not just one.


In [ ]:
LAG, SPAN, HISTORY, MIN_HISTORY = 7, 20, 250, 120
REGIMES = ["calm", "normal", "turbulent"]
REGIME_COLORS = {"calm": "#cde2fb", "normal": "#6da7ec", "turbulent": "#184f95"}

per_day = per_day.sort("Date").with_columns(
    pl.col("std").shift(LAG).rolling_mean(SPAN).alias("trail_disp")   # dates T-26 ... T-7
)
td = per_day["trail_disp"].fill_null(np.nan).to_numpy()
pct = np.full(len(td), np.nan)
for i in range(len(td)):
    past = td[max(0, i - HISTORY):i]
    past = past[~np.isnan(past)]
    if len(past) >= MIN_HISTORY and not np.isnan(td[i]):
        pct[i] = (past < td[i]).mean()      # percentile vs the previous ~1 year
per_day = per_day.with_columns(pl.Series("disp_pct", pct).fill_nan(None)).with_columns(
    pl.when(pl.col("disp_pct").is_null()).then(None)
    .when(pl.col("disp_pct") < 1 / 3).then(pl.lit("calm"))
    .when(pl.col("disp_pct") < 2 / 3).then(pl.lit("normal"))
    .otherwise(pl.lit("turbulent")).alias("regime")
)
day_regime = per_day.select("Date", "regime")
daily = daily.drop("regime", strict=False).join(day_regime, on="Date", how="left")
preds = preds.drop("regime", strict=False).join(day_regime, on="Date", how="left")
test_days = (
    daily.group_by("Date").agg(pl.col("rank_ic").mean(), pl.col("regime").first())
    .sort("Date")
)
print("test days without a regime (too little history):", test_days["regime"].null_count())


def episodes(frame: pl.DataFrame, min_len: int = 5) -> pl.DataFrame:
    """Contiguous runs of the same regime over consecutive test days."""
    runs = (
        frame.with_columns((pl.col("regime") != pl.col("regime").shift()).fill_null(True)
                           .cum_sum().alias("run"))
        .group_by("run", maintain_order=True)
        .agg(pl.col("regime").first(), pl.col("Date").first().alias("start"),
             pl.col("Date").last().alias("end"), pl.len().alias("days"))
    )
    return runs.filter(pl.col("days") >= min_len)


def nw_ci(x: np.ndarray) -> float:
    """Half-width of a 95% CI for mean(x), Newey–West (4 lags)."""
    return 1.96 * abs(x.mean() / newey_west_t(x))


rows = []
eps = episodes(test_days.drop_nulls("regime"))
for r in REGIMES:
    d = daily.filter(pl.col("regime") == r)
    ric = d.sort("Date")["rank_ic"].drop_nans().to_numpy()
    rows.append({
        "regime": r, "days": d["Date"].n_unique(),
        "episodes (>=5 days)": eps.filter(pl.col("regime") == r).height,
        "RankIC": ric.mean(), "RankIC 95% CI ±": nw_ci(ric),
        "IC": d["ic"].mean(), "RankIC>0 %": 100 * (ric > 0).mean(),
        "Q5-Q1": d["q5_minus_q1"].mean(),
    })
by_regime = pl.DataFrame(rows)
by_window = (
    daily.drop_nulls("regime").group_by(["window", "regime"])
    .agg(pl.len().alias("days"), pl.col("rank_ic").mean().alias("RankIC"))
    .sort(["window", "regime"])
)
with pl.Config(float_precision=4, tbl_rows=-1):
    display(by_regime)  # noqa: F821
    display(by_window.pivot(on="regime", index="window", values=["days", "RankIC"]))  # noqa: F821


In [ ]:
def shade(ax) -> None:
    """Shade the background by regime (all runs, including short ones)."""
    runs = episodes(test_days.drop_nulls("regime"), min_len=1)
    starts = runs["start"].to_list()
    ends = starts[1:] + [test_days["Date"].max()]
    for regime_name, s, e in zip(runs["regime"], starts, ends):
        ax.axvspan(s, e, color=REGIME_COLORS[regime_name], alpha=0.45, lw=0)


in_test = per_day.filter(pl.col("Date").is_between(test_days["Date"].min(), test_days["Date"].max()))
# Largest dispersion spikes in the test period, at least 60 days apart.
spikes = []
for row in in_test.sort("std", descending=True).iter_rows(named=True):
    if all(abs((row["Date"] - s).days) > 60 for s in spikes):
        spikes.append(row["Date"])
    if len(spikes) == 3:
        break

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True,
                         gridspec_kw={"height_ratios": [1.2, 0.8, 1.2]})
dates = in_test["Date"].to_numpy()
axes[0].plot(dates, in_test["std"], color="#8f8e88", lw=0.8, label="daily dispersion (label std)")
axes[0].plot(dates, in_test["trail_disp"], color="#0b0b0b", lw=2, label="trailing measure (T-26 ... T-7)")
axes[0].set_title("1. Stock-specific dispersion: the trailing measure reacts only after a spike")
axes[1].plot(dates, in_test["disp_pct"], color="#0b0b0b", lw=1.5)
for level in (1 / 3, 2 / 3):
    axes[1].axhline(level, color="gray", ls="--", lw=1)
axes[1].set_ylim(0, 1)
axes[1].set_ylabel("percentile")
axes[1].set_title("2. Trailing dispersion vs its previous year: < 1/3 calm, > 2/3 turbulent")
t = test_days["Date"].to_numpy()
axes[2].plot(t, test_days["rank_ic"], color="#8f8e88", lw=0.6)
axes[2].plot(t, test_days["rank_ic"].rolling_mean(20, min_samples=5), color=WINDOW_COLORS[0],
             lw=2, label="Rank IC, 20-day mean")
axes[2].axhline(0, color="black", lw=0.8)
axes[2].axhline(test_days["rank_ic"].mean(), color="gray", ls="--", lw=1, label="overall mean")
axes[2].set_title("3. Daily Rank IC against the regime shading")
for ax in axes:
    shade(ax)
    for s in spikes:
        ax.axvline(s, color="#d03b3b", ls=":", lw=1.2)
for s in spikes:
    axes[0].annotate(f"spike {s}", (s, 1.0), xycoords=("data", "axes fraction"),
                     ha="left", va="top", fontsize=8, color="#d03b3b", xytext=(3, -3),
                     textcoords="offset points")
from matplotlib.patches import Patch  # noqa: E402
regime_handles = [Patch(color=REGIME_COLORS[r], alpha=0.45, label=r) for r in REGIMES]
axes[0].legend(loc="upper left", fontsize=8, bbox_to_anchor=(0, 0.9))
axes[2].legend(handles=axes[2].get_legend_handles_labels()[0] + regime_handles, loc="upper left",
               fontsize=8, ncol=5)
save(fig, "regime_timeline")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
xs = np.arange(len(REGIMES))
axes[0].bar(xs, by_regime["RankIC"], color=[REGIME_COLORS[r] for r in REGIMES], width=0.6,
            yerr=by_regime["RankIC 95% CI ±"], capsize=6, ecolor="#52514e")
axes[0].axhline(daily["rank_ic"].mean(), color="gray", ls="--", lw=1, label="all days")
for xi, row in zip(xs, by_regime.iter_rows(named=True)):
    axes[0].annotate(f"{row['days']} days\n{row['episodes (>=5 days)']} episodes", (xi, 0),
                     textcoords="offset points", xytext=(0, 6), ha="center", fontsize=8,
                     color="#0b0b0b" if row["regime"] != "turbulent" else "white")
axes[0].set_xticks(xs, REGIMES)
axes[0].set_title("Mean daily Rank IC by regime (95% Newey–West CI)")
axes[0].legend(loc="upper left", fontsize=8)
for w in range(1, n_windows + 1):
    bw = by_window.filter((pl.col("window") == w) & (pl.col("days") >= 10))
    xw = [REGIMES.index(r) + (w - (n_windows + 1) / 2) * 0.06 for r in bw["regime"]]
    axes[1].plot(xw, bw["RankIC"], "-", color=WINDOW_COLORS[w - 1], lw=1, alpha=0.6)
    axes[1].scatter(xw, bw["RankIC"], s=bw["days"] * 3, color=WINDOW_COLORS[w - 1],
                    edgecolor="white", linewidth=1, label=f"W{w}", zorder=3)
axes[1].axhline(0, color="black", lw=0.8)
axes[1].set_xticks(xs, REGIMES)
axes[1].set_title("Per window (dot size = days; regimes with < 10 days omitted)")
axes[1].legend(ncol=3, fontsize=8, loc="upper left")
save(fig, "regimes")


## Findings: run-2 (reference baseline, windows 1–6), written 2026-10-19

*Written for this run's numbers; other runs need their own reading.*

1. **The signal is consistently real.** Mean daily IC is 0.072–0.133 in every
   window (0.0945 overall), positive on 83–96% of days, and ICIR is 1.15–1.64.
   Newey–West t-stats of 7.8–10.1 per window remain highly significant.
   Daily ICs are strongly autocorrelated (lag-1 ≈ 0.5–0.74, from the overlapping
   labels), so the naive t-stats (and the script's `ir`) are about 1.7×
   overstated.
2. **Pearson IC flatters; Rank IC is the conservative number.** Rank IC is
   0.045–0.090 (0.058 overall), about 0.6× Pearson. Daily Kendall ≈ 0.68 ×
   Rank IC, as expected. The gap means much of the Pearson IC comes from
   extreme stocks.
3. **The edge is mostly in spotting losers.** The bottom prediction quintile
   averages −0.004 to −0.009, while quintiles 2–5 are nearly flat (+0.0003 to
   +0.0045). In A-shares, where shorting is limited, that suits
   excluding stocks more than picking winners.
   *Why one-sided (analysis added 2026-10-20):*
   - **Labels are right-skewed, not dominated by losers:** median −0.005, only
     43% of rows positive, mean ≈ 0. The typical stock slightly
     underperforms, and a few big winners (the fat right tail) lift the mean.
   - **Big winners are unpredictable from these features:** they are spread
     across *all* quintiles, and Q1 holds the most (1.7% of rows > +0.2,
     vs ≈ 0.7% in Q2–Q4). They are mostly news- or theme-driven jumps, so Q5 can't concentrate them and Q2–Q5 look
     alike.
   - **Q1 is the volatile, speculative ("lottery-like") group:** the highest
     volatility (std 0.074 vs ≈ 0.048; trailing-volatility percentile 0.62),
     both tails much fatter (11× more labels < −0.2), yet a negative centre
     (median −0.010, 39% up). Not only tail events: within |label| < 0.1,
     Q1 still averages −0.010 vs −0.003.
   - **The model is neutral; the predictability is one-sided.** With
     short-selling constrained, overpriced or lottery-like stocks stay
     overpriced and drift down, while underpriced ones are bought up quickly
     (Miller 1977). Anomaly returns concentrate on the short side
     (Stambaugh, Yu & Yuan 2012), and investors overpay for lottery-like
     payoffs (the MAX effect, Bali, Cakici & Whitelaw 2011). Our proxies
     are crude and the features opaque: a consistent interpretation, not
     proof. It holds for *this* feature set; other information (news,
     flows) might predict ups.
   - **The predictions are not pessimistic:** mean 0.0002 (= label mean),
     median **+0.0015**, only 22% negative, skew −6.2 (1% / 99% quantiles
     −0.026 / +0.006). Mildly positive for most stocks, with a long, confident
     negative tail for a minority. The median prediction exceeds the median
     label because squared-error loss predicts the *mean*, and the mean
     includes the rare jackpots.
   - **Implications:** long-only use means "avoid Q1". Better upside needs
     other information (news, attention, flows; maybe history features).
     Clipping labels would cut the jackpot noise from Q1 in training.
4. **Calibration is decent in the bulk and off in the tails.** The most
   negative predictions overshoot (W4 badly). Upside predictions barely
   materialize.
5. **Val R² tracks test R² in W1–W3, but not W4–W5** (val 0.013–0.015 vs test
   0.002–0.004), while IC stays fine there. The miss is in magnitudes, not
   ordering. It is either a regime shift in those test periods (label spread
   spikes in early 2024 and autumn 2024) or the I-1 leak: a
   chronological-validation run can separate the two.
6. **Pooling barely matters for Kendall** (pooled `kt` ≈ daily Kendall),
   because the label's daily cross-sectional mean is ≈ 0 (std over days
   0.0027): the factor adjustment demeans it. So I-4 matters little in
   practice.
7. **Label scale (I-8):** std 0.057, robust std 0.031, excess kurtosis 22.7.
   Very heavy tails.
   *Label meaning (2026-10-21):* treat the label as roughly a 5-day forward
   residual return (Barra-type factor model), without attaching
   further financial meaning. Values are reported in label units. The open
   scale details are not pursued further.
8. **Cost:** 22–42 s and ≤ 2.4 GB per window, about 3 min for all six.

### Finer grain (section 5, added 2026-10-20)

9. **Window results aren't driven by a few days.** The median and 10%-trimmed
   mean of daily IC are within ±0.01 of the mean in every window; Rank IC is
   positive on 79–90% of days.
10. **Overconfident months (slope < 0.5) occur in every window**, about one to
    three times each: 2023-06, 2023-09, 2024-02, 2024-09/10, 2025-02/04/05.
    W4 and W5 simply had more of them, and no compensating strong months (W3
    had Mar–Jun 2024, Rank IC ≈ 0.13).
11. **Pearson and Rank IC can disagree within a month**, e.g. 2024-01: Pearson
    0.088 vs Rank −0.029; 2023-05: 0.122 vs 0.009. In those months the signal
    works only through the extreme stocks.
12. **Ex-ante regimes (section 6).** Using one-year rolling percentiles of
    trailing dispersion, mean daily Rank IC is 0.042 on calm days (283 days,
    8 episodes), 0.053 on normal days (242, 11) and **0.088 on turbulent days**
    (195 days, but only **3 episodes**), with 95% CIs ±0.012 / ±0.011 / ±0.018.
    - Calm vs normal: no real difference (the CIs overlap).
    - Turbulent > the rest, but most of it comes from two episodes: the weeks
      after the Feb-2024 and Sep-2024 spikes (W3 0.119, W4 0.075). W1's
      turbulent stretch (May–Jun 2023) shows the opposite (0.002). Treat it as
      a hint that *the aftermath of a dispersion spike is a good time for this
      signal*, not as an established effect.
    - The regime can't warn of shocks: spike onsets (2024-02-07, 2024-09-23,
      2025-03-31) fall in calm or normal stretches, and the worst IC dips
      (late Jan / early Feb 2024) were classed calm. So it is no tool for
      avoiding bad periods.
    - Use for later: compare history models vs the baseline **within each
      regime**, to see whether history mainly helps after shocks.


## Summary: what the baseline achieves, and what to improve

**Setup.** Six rolling LightGBM models (single-day features, the author's
original protocol), each tested on the next ~120 trading days: 720 test days,
2022-12-29 → 2025-12-18. The final window is held back.

**Overall performance.** A stable, statistically solid signal.
- Mean daily **Rank IC 0.058** (Pearson IC 0.0945), ICIR 1.34, Pearson IC
  positive on 92% of days; Newey–West t ≈ 21 overall, 8–10 per window.
- Every window positive (Rank IC 0.045–0.090). Medians ≈ means, so no window
  is carried by a few lucky days.
- Top-minus-bottom quintile spread ≈ 0.005–0.013 in label units (gross; not
  a profit).
- Cheap: ~30 s and ≤ 2.4 GB per window.

**Where it excels.**
- **Spotting losers:** the bottom prediction quintile averages −0.004 to
  −0.009 (label units); quintiles 2–5 are nearly flat. Downs are more systematic (a plausible
  reading: volatile, lottery-like stocks stay overpriced under A-share
  shorting limits and drift down). Ups are dominated by news- or theme-driven jumps the features can't
  see. The model isn't pessimistic: it is mildly positive on ~4/5 of stocks
  (median prediction +0.0015) and confidently negative on a minority.
- **Ordering, consistently:** strong stretches such as Mar–Jun 2024 and
  Nov–Dec 2024 (Rank IC ≈ 0.10–0.13).
- **After dispersion spikes:** Rank IC ≈ 0.09 in "turbulent" (post-spike)
  regimes. A hint only, from 3 episodes.

**Where it stumbles.**
- **Magnitudes:** test R² is only 0.002–0.017. Predictions are overconfident
  (calibration slope < 0.5) in 1–3 months per window, worst in W4/W5
  (2024-09/10, 2025-02/04/05). Large predicted gains rarely materialize.
  This matters as much as ranking: the predictions feed a portfolio
  optimizer, which uses their magnitudes.
- **Shock onsets:** e.g. Jan–Feb 2024, when Rank IC turned negative. A
  trailing indicator can't see them coming.
- **Some months work only through extreme stocks** (Pearson ≫ Rank IC, e.g.
  2024-01).

**Label distribution and why it matters.**
- **What the label is:** roughly a 5-day forward residual return (Barra-type
  factor model). No further financial
  meaning is attached to label or prediction values; they are reported in
  label units. Daily mean ≈ 0.
- **Fat tails:** std 0.057 vs robust std 0.031, excess kurtosis 22.7.
  |label| > 0.2 in 1.2% of rows, > 0.5 in 0.07%. Extremes are mostly limit-up
  streaks and new listings, often untradeable.
- **Training:** squared error (plus GOSS, which keeps large-error rows) gives
  these rare rows outsized weight, which leads to overconfident tail
  predictions.
- **Evaluation:** they inflate Pearson IC (≈ 1.6× Rank IC) and dominate R² and
  RMSE. Rank IC is the robust ranking measure; magnitudes are judged by
  calibration slope, R² and RMSE. Both matter, since the predictions feed a
  portfolio optimizer.

**Probable reasons for the failures.**
1. **The test period itself**, not the training: models trained on calm and
   on turbulent periods fail in the *same* months by the same amount. The
   feature → label relationship weakens during regime shifts and theme
   rotations.
2. **Fat-tailed labels with squared-error loss** → overconfident magnitudes.
3. **No market-state information:** a single-day, pooled model can't tell
   which conditions it is in.
4. **Unforeseeable shocks:** the onsets are invisible ex-ante.

The validation leak (I-1) looks small (val R² ≈ test R² in W1–W3); the
W4/W5 gaps come from the test side.

**What the baseline has achieved.** A reproducible, trustworthy reference:
consistently positive Rank IC over three years, with complete records
(models, predictions, curves) and an analysis pipeline. It is a solid
benchmark for the history experiments: even +0.005 Rank IC would be a real
gain if it holds across windows, provided calibration doesn't get worse.

**What to improve next.**
1. **Protocol:** chronological validation with a purge gap; feature
   selection on training dates only (I-1). Report Rank IC, unscaled ICIR and
   Newey–West t (I-4, I-5).
2. **Magnitudes (as important as ranking, for the optimizer):** per-day
   demeaning of predictions and shrinkage fitted on validation data.
3. **Tails:** per-day label clipping or Huber loss; exclude new listings.
   Rank-Gauss labels would discard magnitudes, so they are less suitable
   here (N-10).
4. **Training settings:** `min_data_in_leaf`, GOSS vs bagging, `reg_alpha`
   (I-2).
5. **Project goal:** history features, compared **paired by day**, and also
   within months and regimes. History may help most around regime changes.
